# Lesson 11 Lab — GPTQ: Second-Order Intuition and Layer Reconstruction

**Puzzle:** Why should two weights with the same magnitude receive different quantization treatment?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

GPTQ reconstructs one layer at a time using the layer weights and representative input activations. It targets output distortion, not unweighted distance between original and rounded weights.

### Core mechanism

For weight error `ΔW` and inputs `X`, layer error is approximately `||XΔWᵀ||²`; the input Gram/Hessian approximation `XᵀX` weights sensitive directions. GPTQ uses inverse-Hessian information to compensate remaining weights as columns are quantized.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "11-gptq"
device = require_cuda()
torch.manual_seed(2026 + 11)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Block size and damping control memory, numerical stability, and approximation quality. Ordering and calibration data alter the result, and the packed inference kernel is a separate concern.

### What this code tests

The lab is deliberately GPTQ-inspired: it uses input-weighted sensitivity and a fallback to expose the objective, while clearly not claiming GPTQModel execution.

**Experiment:** Compare naive INT4 weight quantization with a GPTQ-inspired sensitivity fallback that preserves columns with large input-weighted error.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
n,in_f,out_f=1024,256,192; x=torch.randn(n,in_f,device=device); x[:,::31]*=5; w=torch.randn(out_f,in_f,device=device)
ref=x@w.t(); _,_,naive=symmetric_quantize(w,bits=4,group_size=64); naive_out=x@naive.t()
sensitivity=x.square().mean(0)*((w-naive).square().mean(0)); keep=torch.topk(sensitivity,k=in_f//8).indices
aware=naive.clone(); aware[:,keep]=w[:,keep]; aware_out=x@aware.t()
result=base_result(11,"numerical-model"); result.update({"shape":[n,in_f,out_f],"preserved_column_fraction":round(len(keep)/in_f,4),
    "naive_output_error":error_metrics(ref,naive_out),"sensitivity_fallback_error":error_metrics(ref,aware_out),
    "conclusion":"Input-weighted sensitivity changed which quantization errors mattered; this is a GPTQ intuition model, not GPTQModel execution."})


## 3. Inspect the evidence

Measure layer-output error on held-out inputs and label the experiment as an intuition model, not a GPTQ kernel benchmark.

### Acceptance and rollback gate

Record calibration activations, damping, block/group size, ordering, layer reconstruction loss, end-task regression, and the deployed operator.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Input-weighted sensitivity changed which quantization errors mattered; this is a GPTQ intuition model, not GPTQModel execution.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:34+00:00",
  "lesson": 11,
  "naive_output_error": {
    "cosine": 0.99448818,
    "mae": 1.84227121,
    "max_abs": 12.09428406,
    "rmse": 2.32480502
  },
  "preserved_column_fraction": 0.125,
  "schema_version": 1,
  "sensitivity_fallback_error": {
    "cosine": 0.99738872,
    "mae": 1.26920569,
    "max_abs": 8.99761105,
    "rmse": 1.59714472
  },
  "shape": [
    1024,
    256,
    192
  ]
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Second-order information changes the objective from nearest weights to faithful layer outputs.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).